# 05_a — Série histórica dos repasses da SEFAZ/RN (2004–2025)

**Objetivo:** ampliar a série de repasses obrigatórios aos municípios (hoje 2020–2023) para o maior período
disponível no site da SEFAZ/RN, baixando tudo automaticamente, sem salvar arquivo por arquivo.

**Fonte:** <https://www.sefaz.rn.gov.br/documentos/Transparência/Transferências para os Municípios do RN/>

O site tem uma pasta por ano e os arquivos vêm em **dois formatos**:

| Anos | Formato | Como tratamos |
|---|---|---|
| 2019 em diante | Planilha `.xls` (o mesmo formato do notebook 01) | Lidas com a mesma função `ler_aba` |
| 2004 a 2018 | **Só PDF**, um arquivo por ano | Extração do texto e conversão para tabela (seções 4 e 5) |

**Padrão mantido em todos os anos:** valor **anual** por município de **ICMS, IPVA, IPI e ROYALTIES**.
A CIDE, que aparece nos PDFs antigos e deixou de ser repassada, fica de fora.

**Etapas do notebook**
1. Bibliotecas e funções reaproveitadas do notebook 01
2. Mapa do site: quais arquivos existem e qual usar em cada ano
3. Planilhas `.xls` (2019 em diante)
4. PDFs (2004 a 2018): download
5. PDFs → tabela
6. Junção de planilhas + PDFs e conferência
7. Saída `repasses_2004_2025.csv`

**Sequência da série longa:** este notebook (05_a) gera a série de repasses; o **05_b** coleta os indicadores
(população, PIB, CadÚnico, IPCA) para os mesmos anos; o **05_c** junta os dois e faz as análises e gráficos.

## 1. Bibliotecas e funções reaproveitadas

A lista oficial de municípios do IBGE, a função `chave()` e o `DE_PARA` são os mesmos do notebook 01:
é assim que cada nome de município das planilhas e PDFs vira o código IBGE.

In [ ]:
import os, re, html, unicodedata
from urllib.parse import unquote, quote

import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_colwidth', 90)

PASTA = 'dados/sefaz_site'                 # onde os arquivos baixados do site ficam no Colab
os.makedirs(f'{PASTA}/xls', exist_ok=True)
os.makedirs(f'{PASTA}/pdf', exist_ok=True)
os.makedirs('saida', exist_ok=True)

TIPOS = ['ICMS', 'IPVA', 'IPI', 'ROYALTIES']   # repasses mantidos em todos os anos
ANO_ULTIMO_COMPLETO = 2025                     # 2026 ainda está em andamento, fica de fora
HEADERS = {'User-Agent': 'Mozilla/5.0'}        # o site da SEFAZ recusa pedidos sem identificação de navegador

REPO = 'Renata-Nascimento/Projeto-U02-CD'
RAW_GITHUB = f'https://raw.githubusercontent.com/{REPO}/main/'

### Lista de municípios: API do IBGE, com cópia no GitHub como plano B

O Colab roda em servidores **fora do Brasil**, e os serviços do IBGE às vezes não respondem a conexões vindas
do exterior (o erro aparece como `ConnectTimeout`). Por isso o código tenta a API por 20 segundos e, se não
conseguir, lê a cópia da mesma lista guardada no repositório (`dados/ibge/municipios_rn.csv`).

In [ ]:
try:
    resp = requests.get('https://servicodados.ibge.gov.br/api/v1/localidades/estados/24/municipios', timeout=20)
    resp.raise_for_status()
    municipios = pd.DataFrame([{'cod_ibge': m['id'], 'municipio': m['nome']} for m in resp.json()])
    print('Lista de municípios: API do IBGE')
except requests.exceptions.RequestException as erro:
    print(f'API do IBGE não respondeu ({erro.__class__.__name__}); usando a cópia do GitHub')
    municipios = pd.read_csv(RAW_GITHUB + 'dados/ibge/municipios_rn.csv')[['cod_ibge', 'municipio']]
print(len(municipios), 'municípios')

def chave(nome):
    """'Olho d'Água do Borges' -> 'OLHODAGUADOBORGES'"""
    s = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode().upper()
    return re.sub(r'[^A-Z]', '', s)

DE_PARA = {
    'ARES': 'AREZ',
    'BOASAUDE': 'JANUARIOCICCO',                 # nome popular (oficial: Januário Cicco)
    'CAICARADORIODOSVENTOS': 'CAICARADORIODOVENTO',
    'GOVDIXSEPTROSADO': 'GOVERNADORDIXSEPTROSADO',
    'IPANGUASSU': 'IPANGUACU',
    'OLHODAGUADOSBORGES': 'OLHODAGUADOBORGES',
    'RUIBARBOSA': 'RUYBARBOSA',
    'SAOMIGUELDETOUROS': 'SAOMIGUELDOGOSTOSO',   # nome antigo
    'SENELOIDESOUZA': 'SENADORELOIDESOUZA',
    'SENGEORGINOAVELINO': 'SENADORGEORGINOAVELINO',
}

COD_POR_CHAVE = dict(zip(municipios['municipio'].map(chave), municipios['cod_ibge']))

API do IBGE não respondeu (ConnectTimeout); usando a cópia do GitHub
167 municípios


## 2. Mapa do site da SEFAZ

A página principal lista as pastas (uma por ano, mais "2018 e anos anteriores"). Cada pasta lista os arquivos,
e cada arquivo tem um link de download direto (`.../wp-json/nextcloud/v1/download?filePath=...`).
O código lê essas páginas e monta uma tabela com todos os arquivos encontrados.

**Plano B:** se o site da SEFAZ não responder (por exemplo, por bloquear conexões vindas do Colab, que roda fora
do Brasil), a tabela é montada a partir da cópia dos arquivos guardada no repositório (`dados/sefaz`).

In [ ]:
BASE_SITE = 'https://www.sefaz.rn.gov.br/documentos/Transparência/Transferências para os Municípios do RN/'

def links_da_pagina(url):
    """Devolve todos os links (href) de uma página, já decodificados (%20 -> espaço, &amp; -> &).
    timeout=(15, 90): até 15 s para conectar e até 90 s para receber a página."""
    pagina = requests.get(url, timeout=(15, 90), headers=HEADERS)
    pagina.raise_for_status()
    return [unquote(html.unescape(h)) for h in re.findall(r'href="([^"]+)"', pagina.text)]

def arquivos_no_site():
    """Lê a página principal (pastas) e cada pasta (arquivos com link de download)."""
    pastas = sorted({l for l in links_da_pagina(BASE_SITE)
                     if l.startswith(BASE_SITE) and l != BASE_SITE and l.endswith('/')})
    registros = []
    for pasta in pastas:
        for link in sorted(set(links_da_pagina(pasta))):
            if 'nextcloud/v1/download' in link:
                registros.append({'pasta': pasta[len(BASE_SITE):].strip('/'),
                                  'arquivo': link.split('/')[-1], 'url': link})
    return pd.DataFrame(registros)

def arquivos_no_github():
    """Lista os .xls e .pdf da pasta dados/sefaz do repositório (API do GitHub), com o link de download."""
    arvore = requests.get(f'https://api.github.com/repos/{REPO}/git/trees/main?recursive=1', timeout=60).json()['tree']
    registros = []
    for item in arvore:
        caminho_repo = item['path']
        if caminho_repo.startswith('dados/sefaz/') and caminho_repo.lower().endswith(('.xls', '.pdf')):
            registros.append({'pasta': 'GitHub: ' + caminho_repo.rsplit('/', 1)[0],
                              'arquivo': caminho_repo.rsplit('/', 1)[1],
                              'url': RAW_GITHUB + quote(caminho_repo)})
    return pd.DataFrame(registros)

try:
    site = arquivos_no_site()
    print('Mapa montado a partir do site da SEFAZ')
except requests.exceptions.RequestException as erro:
    print(f'Site da SEFAZ não respondeu ({erro.__class__.__name__}); usando a cópia do GitHub')
    site = arquivos_no_github()

# informações tiradas do nome do arquivo
site['formato'] = site['arquivo'].str.extract(r'\.(\w+)$', expand=False).str.lower()
site['ano'] = site['arquivo'].str.extract(r'(20\d{2})', expand=False).astype('Int64')

def data_referencia(nome):
    """'... até o dia 30-12-2019.xls' ou '(... ate o dia 30-12-04).pdf' -> Timestamp('2019-12-30')."""
    achados = re.findall(r'(\d{2})-(\d{2})-(\d{2,4})', nome)
    if not achados:
        return pd.NaT
    dia, mes, ano = achados[-1]
    ano = int(ano) + 2000 if len(ano) == 2 else int(ano)
    return pd.Timestamp(year=ano, month=int(mes), day=int(dia))

site['data_referencia'] = site['arquivo'].map(data_referencia)
print(len(site), 'arquivos encontrados em', site['pasta'].nunique(), 'pastas')
site[['pasta', 'arquivo', 'formato', 'ano', 'data_referencia']]

Site da SEFAZ não respondeu (ConnectionError); usando a cópia do GitHub
22 arquivos encontrados em 2 pastas


,pasta,arquivo,formato,ano,data_referencia
0,GitHub: dados/sefaz/pdf,repasses_2004.pdf,pdf,2004,NaT
1,GitHub: dados/sefaz/pdf,repasses_2005.pdf,pdf,2005,NaT
2,GitHub: dados/sefaz/pdf,repasses_2006.pdf,pdf,2006,NaT
3,GitHub: dados/sefaz/pdf,repasses_2007.pdf,pdf,2007,NaT
4,GitHub: dados/sefaz/pdf,repasses_2008.pdf,pdf,2008,NaT
5,GitHub: dados/sefaz/pdf,repasses_2009.pdf,pdf,2009,NaT
6,GitHub: dados/sefaz/pdf,repasses_2010.pdf,pdf,2010,NaT
7,GitHub: dados/sefaz/pdf,repasses_2011.pdf,pdf,2011,NaT
8,GitHub: dados/sefaz/pdf,repasses_2012.pdf,pdf,2012,NaT
9,GitHub: dados/sefaz/pdf,repasses_2013.pdf,pdf,2013,NaT


### Qual arquivo usar em cada ano

O site tem armadilhas que fariam os valores serem **somados mais de uma vez**:
- **2023** tem duas planilhas com o mesmo conteúdo (o nome muda só numa letra maiúscula);
- **2025** tem 10 planilhas: cada uma acumula o ano **até** uma data (janeiro, fevereiro... dezembro);
- a pasta "2018 e anos anteriores" tem PDFs até 2023, mas de 2019 em diante já temos as planilhas;
- o PDF de 2021 vai só até outubro;
- a pasta 2026 tem outros relatórios (arrecadação do IPVA, "Transferências Municipais") e o ano está incompleto.

**Regras:**
1. só arquivos cujo nome começa com "Repasses" (maiúscula ou minúscula);
2. planilhas `.xls` para **2019 a 2025**; PDFs só para os anos **sem planilha** (2004 a 2018);
3. em cada ano, fica **um único arquivo: o de data de referência mais recente**, que é o acumulado do ano inteiro.

In [ ]:
candidatos = site[site['arquivo'].str.lower().str.startswith('repasses')].copy()

anos_xls = set(candidatos.loc[candidatos['formato'] == 'xls', 'ano'].dropna())
candidatos = candidatos[
    ((candidatos['formato'] == 'xls') & (candidatos['ano'] <= ANO_ULTIMO_COMPLETO)) |
    ((candidatos['formato'] == 'pdf') & ~candidatos['ano'].isin(anos_xls))
]

escolhidos = (candidatos.sort_values(['ano', 'data_referencia'])
              .groupby('ano').tail(1)            # o mais recente de cada ano
              .sort_values('ano').reset_index(drop=True))

descartados = site[~site['url'].isin(escolhidos['url'])]
print(f'{len(escolhidos)} arquivos escolhidos ({escolhidos["ano"].min()}–{escolhidos["ano"].max()}), '
      f'{len(descartados)} descartados')
escolhidos[['ano', 'formato', 'data_referencia', 'arquivo']]

22 arquivos escolhidos (2004–2025), 0 descartados


,ano,formato,data_referencia,arquivo
0,2004,pdf,NaT,repasses_2004.pdf
1,2005,pdf,NaT,repasses_2005.pdf
2,2006,pdf,NaT,repasses_2006.pdf
3,2007,pdf,NaT,repasses_2007.pdf
4,2008,pdf,NaT,repasses_2008.pdf
5,2009,pdf,NaT,repasses_2009.pdf
6,2010,pdf,NaT,repasses_2010.pdf
7,2011,pdf,NaT,repasses_2011.pdf
8,2012,pdf,NaT,repasses_2012.pdf
9,2013,pdf,NaT,repasses_2013.pdf


In [ ]:
# Conferência: um arquivo por ano, sem buracos na sequência
anos = escolhidos['ano'].tolist()
assert len(anos) == len(set(anos)), 'há ano repetido'
assert anos == list(range(min(anos), max(anos) + 1)), 'há ano faltando'
print('OK: um arquivo por ano, de', min(anos), 'a', max(anos))

OK: um arquivo por ano, de 2004 a 2025


## 3. Planilhas `.xls` (2019 a 2025)

### 3.1 Download: primeiro do GitHub, e do site da SEFAZ só se faltar

O servidor da SEFAZ é lento (cerca de 1 minuto por arquivo). Por isso, uma cópia de cada arquivo escolhido na
seção 2 fica guardada no repositório do grupo, na pasta `dados/sefaz` (os PDFs em `dados/sefaz/pdf`).

Para cada ano, o código procura o arquivo no GitHub (segundos). Só se ele **não estiver lá** é que baixa do site
da SEFAZ. Assim o notebook roda rápido e, quando a SEFAZ publicar um ano novo, ele é baixado do site
automaticamente; depois é só subir a cópia no GitHub.

O link original de cada arquivo no site da SEFAZ continua registrado na tabela `escolhidos` (seção 2).

In [ ]:
import time

# {(ano, formato): link de download} dos arquivos guardados no repositório
copia_github = arquivos_no_github()
no_github = {(int(re.search(r'(20\d{2})', n).group(1)), n.rsplit('.', 1)[1].lower()): u
             for n, u in zip(copia_github['arquivo'], copia_github['url']) if re.search(r'(20\d{2})', n)}
print(len(no_github), 'arquivos da SEFAZ encontrados no GitHub')

def obter_arquivo(linha, destino):
    """Baixa o arquivo do ano: do GitHub se houver cópia lá, senão do site da SEFAZ.
    Se o arquivo já estiver na pasta do Colab (mesma sessão), não baixa de novo."""
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return 'já estava na pasta'
    url_github = no_github.get((int(linha['ano']), linha['formato']))
    url, origem = (url_github, 'GitHub') if url_github else (linha['url'], 'site da SEFAZ')
    inicio = time.time()
    r = requests.get(url, timeout=180, headers=HEADERS)
    r.raise_for_status()
    with open(destino, 'wb') as f:
        f.write(r.content)
    return f'{origem}, {time.time() - inicio:.0f}s'

planilhas = escolhidos[escolhidos['formato'] == 'xls'].copy()
planilhas['caminho'] = [f'{PASTA}/xls/repasses_{a}.xls' for a in planilhas['ano']]
planilhas['origem'] = [obter_arquivo(linha, linha['caminho']) for _, linha in planilhas.iterrows()]
planilhas[['ano', 'arquivo', 'origem']]

22 arquivos da SEFAZ encontrados no GitHub


,ano,arquivo,origem
15,2019,repasses_2019.xls,"GitHub, 0s"
16,2020,repasses_2020.xls,"GitHub, 0s"
17,2021,repasses_2021.xls,"GitHub, 0s"
18,2022,repasses_2022.xls,"GitHub, 0s"
19,2023,repasses_2023.xls,"GitHub, 0s"
20,2024,repasses_2024.xls,"GitHub, 0s"
21,2025,repasses_2025.xls,"GitHub, 0s"


### 3.2 Leitura

Mesma lógica da `ler_aba` do notebook 01 (cabeçalho pela palavra `MUNICÍPIOS`, meses pelo nome, só linhas que
casam com a lista do IBGE, parada no TOTAL), com dois ajustes descobertos ao ampliar os anos:

- **Parcela "INDICE <ano anterior> JANEIRO" do ICMS:** a partir de 2021 o ICMS tem uma coluna extra com uma
  parcela de janeiro calculada pelo índice do ano anterior. O nome não é exatamente "JANEIRO", então antes ela
  ficava de fora (1% a 2% do ICMS do ano). Agora uma coluna conta como mês quando o nome **termina** com o mês.
- **Meses uma linha acima:** em ROYALTIES 2019 os nomes dos meses estão na linha de cima do "Municípios".
  Agora o código procura o mês nas duas linhas do cabeçalho.

E duas checagens novas:

- **soma dos meses × coluna TOTAL da própria planilha:** garante que pegamos o **acumulado do ano** e não só
  um mês (a preocupação com as planilhas de 2025);
- **nomes que não casaram com o IBGE:** linhas com valores numéricos cujo nome não foi reconhecido. Se aparecer
  algum, ele precisa entrar no `DE_PARA`.

In [ ]:
MESES = ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO',
         'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']

def eh_mes(texto_coluna):
    """'JANEIRO' -> True; 'INDICEJANEIRO' (parcela 'INDICE 2020 JANEIRO') -> True; 'TOTAL' -> False."""
    return any(texto_coluna.endswith(m) for m in MESES)

def ler_aba(arquivo, aba, ano):
    bruto = pd.read_excel(arquivo, sheet_name=aba, header=None)
    lin_cab = bruto.index[bruto[0].astype(str).str.upper().str.startswith('MUNIC')][0]
    cabecalho = [chave(c) for c in bruto.iloc[lin_cab]]
    cabecalho_acima = [chave(c) for c in bruto.iloc[lin_cab - 1]]   # meses ou "TOTAL" às vezes ficam uma linha acima
    corpo = bruto.iloc[lin_cab + 1:]

    k = corpo[0].map(chave).replace(DE_PARA)
    inicio = k[k.isin(list(COD_POR_CHAVE))].index[0]
    fim = corpo.index[(corpo.index > inicio) & corpo[0].astype(str).str.upper().str.contains('TOTA')]
    if len(fim):
        corpo, k = corpo.loc[:fim[0] - 1], k.loc[:fim[0] - 1]

    colunas_meses = [j for j, (c, ca) in enumerate(zip(cabecalho, cabecalho_acima)) if eh_mes(c) or eh_mes(ca)]
    valores = corpo[colunas_meses].apply(pd.to_numeric, errors='coerce')

    # nomes com valores que não casaram com o IBGE (a partir do 1º município)
    tem_valor = valores.notna().any(axis=1) & (corpo.index >= inicio)
    nao_casados = corpo.loc[tem_valor & ~k.isin(list(COD_POR_CHAVE)), 0].astype(str).str.strip().tolist()

    casou = k.isin(list(COD_POR_CHAVE))
    tabela = pd.DataFrame({'cod_ibge': k[casou].map(COD_POR_CHAVE).astype(int).values,
                           'nome_na_fonte': corpo.loc[casou, 0].astype(str).str.strip().values,
                           'ano': ano, 'tipo': aba,
                           'valor': valores[casou].sum(axis=1).values})

    # coluna TOTAL da planilha, se existir
    col_total = [j for j, (c, ca) in enumerate(zip(cabecalho, cabecalho_acima)) if 'TOTAL' in (c, ca)]
    total_planilha = (pd.to_numeric(corpo.loc[casou, col_total[-1]], errors='coerce').sum()
                      if col_total else np.nan)

    diagnostico = {'ano': ano, 'tipo': aba, 'municipios': tabela['cod_ibge'].nunique(),
                   'colunas_de_mes': len(colunas_meses),
                   'soma_dos_meses': tabela['valor'].sum(), 'coluna_TOTAL': total_planilha,
                   'nao_casados': nao_casados}
    return tabela, diagnostico

partes, diagnosticos = [], []
for _, linha in planilhas.iterrows():
    for aba in TIPOS:
        tabela, diag = ler_aba(linha['caminho'], aba, int(linha['ano']))
        partes.append(tabela)
        diagnosticos.append(diag)

repasses_xls = pd.concat(partes, ignore_index=True)
repasses_xls['formato_fonte'] = 'xls'
diag_xls = pd.DataFrame(diagnosticos)
print(repasses_xls.shape)

(4676, 6)


### 3.3 Conferência das planilhas

O que cada coluna da tabela abaixo mostra:
- `municipios`: quantos municípios foram lidos na aba (o certo é 167);
- `colunas_de_mes`: quantas colunas de mês foram encontradas (12, ou 13–14 quando há a parcela "INDICE ano
  anterior JANEIRO" ou um mês repetido);
- `soma_R_milhoes`: total do ano calculado pelo código (soma dos meses);
- `TOTAL_R_milhoes`: total do ano informado pela própria SEFAZ, na coluna TOTAL da planilha;
- `diferenca_%`: diferença entre os dois totais. **Zero** significa que o código pegou todos os meses.

Os nomes de municípios não reconhecidos não aparecem na tabela: eles são verificados na célula seguinte, que
interrompe o notebook e mostra os nomes se algum precisar entrar no `DE_PARA`.

In [ ]:
diag_xls['diferenca_%'] = (diag_xls['soma_dos_meses'] / diag_xls['coluna_TOTAL'] - 1) * 100
(diag_xls.assign(soma_R_milhoes=diag_xls['soma_dos_meses'] / 1e6,
                 TOTAL_R_milhoes=diag_xls['coluna_TOTAL'] / 1e6)
 [['ano', 'tipo', 'municipios', 'colunas_de_mes', 'soma_R_milhoes', 'TOTAL_R_milhoes', 'diferenca_%']]
 .round(2))

,ano,tipo,municipios,colunas_de_mes,soma_R_milhoes,TOTAL_R_milhoes,diferenca_%
0,2019,ICMS,167,13,1388.86,1388.86,0.0
1,2019,IPVA,167,12,204.30,204.30,0.0
2,2019,IPI,167,12,1.22,1.22,0.0
3,2019,ROYALTIES,167,12,26.53,26.53,0.0
4,2020,ICMS,167,12,1414.54,1414.54,0.0
5,2020,IPVA,167,12,197.83,197.83,0.0
6,2020,IPI,167,12,1.49,1.49,0.0
7,2020,ROYALTIES,167,12,20.24,20.24,0.0
8,2021,ICMS,167,13,1669.50,1669.50,0.0
9,2021,IPVA,167,12,218.99,218.99,0.0


In [ ]:
# Checagens automáticas: se alguma falhar, o notebook para aqui e mostra o problema
sem_correspondencia = sorted({nome for lista in diag_xls['nao_casados'] for nome in lista})
assert not sem_correspondencia, f'nomes de municípios fora do DE_PARA: {sem_correspondencia}'
assert (diag_xls['municipios'] == 167).all(), 'alguma aba não tem os 167 municípios'
assert (diag_xls['colunas_de_mes'] >= 12).all(), 'alguma aba não tem os 12 meses'
assert (diag_xls['diferenca_%'].abs() < 0.01).all(), 'soma dos meses diferente da coluna TOTAL da planilha'
assert not repasses_xls.duplicated(['cod_ibge', 'ano', 'tipo']).any(), 'há município repetido'
print('OK: 167 municípios, 12 meses, soma igual ao TOTAL da planilha e nenhum nome sem correspondência')

OK: 167 municípios, 12 meses, soma igual ao TOTAL da planilha e nenhum nome sem correspondência


In [ ]:
# Grafias diferentes do mesmo município encontradas nas planilhas (todas resolvidas pela chave + DE_PARA)
grafias = repasses_xls.groupby('cod_ibge')['nome_na_fonte'].unique()
grafias = grafias[grafias.str.len() > 1].map(lambda v: ' | '.join(sorted(v)))
print(len(grafias), 'municípios aparecem com mais de uma grafia')
grafias.to_frame('grafias encontradas')

15 municípios aparecem com mais de uma grafia


,grafias encontradas
cod_ibge,
2401859,CAICARA DO NORTE | CAIÇARA DO NORTE
2401909,CAICARA DO RIO DO VENTO | CAICARA DO RIO DOS VENTOS
2402600,CEARA MIRIM | CEARA-MIRIM
2402709,CERRO CORA | CERRO-CORA
2404309,GOV. DIX-SEPT ROSADO | GOVERNADOR DIX-SEPT ROSADO
2404705,IPANGUACU | IPANGUASSU
2404853,ITAJA | ITAJÁ
2406155,JUNDIA | JUNDIÁ
2406205,LAGOA DANTA | LAGOA D`ANTA


In [ ]:
# Total anual por tipo (R$ milhões, valores nominais)
(repasses_xls.pivot_table(index='ano', columns='tipo', values='valor', aggfunc='sum') / 1e6).round(1)

tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2019,1388.9,1.2,204.3,26.5
2020,1414.5,1.5,197.8,20.2
2021,1669.5,1.6,219.0,35.5
2022,1766.0,2.2,250.0,52.0
2023,2100.2,4.3,307.0,41.7
2024,2076.0,4.1,306.4,41.2
2025,2317.7,9.5,316.3,40.4


## 4. PDFs (2004 a 2018): download

Para os anos sem planilha, o site só tem PDF: um arquivo por ano, já escolhido na seção 2.
Mesma regra da seção 3.1: cópia do GitHub (`dados/sefaz/pdf`) e, se faltar, o site da SEFAZ
(que leva cerca de 1 minuto por PDF).

In [ ]:
pdfs = escolhidos[escolhidos['formato'] == 'pdf'].copy()
pdfs['caminho'] = [f'{PASTA}/pdf/repasses_{a}.pdf' for a in pdfs['ano']]
pdfs['origem'] = [obter_arquivo(linha, linha['caminho']) for _, linha in pdfs.iterrows()]
pdfs['tamanho_KB'] = [round(os.path.getsize(c) / 1024) for c in pdfs['caminho']]
pdfs[['ano', 'arquivo', 'origem', 'tamanho_KB']]

,ano,arquivo,origem,tamanho_KB
0,2004,repasses_2004.pdf,"GitHub, 0s",83
1,2005,repasses_2005.pdf,"GitHub, 0s",72
2,2006,repasses_2006.pdf,"GitHub, 0s",71
3,2007,repasses_2007.pdf,"GitHub, 0s",71
4,2008,repasses_2008.pdf,"GitHub, 0s",78
5,2009,repasses_2009.pdf,"GitHub, 0s",75
6,2010,repasses_2010.pdf,"GitHub, 0s",81
7,2011,repasses_2011.pdf,"GitHub, 0s",76
8,2012,repasses_2012.pdf,"GitHub, 0s",387
9,2013,repasses_2013.pdf,"GitHub, 1s",376


### 4.1 O que tem dentro de cada PDF

Usamos a biblioteca `pdfplumber`, que lê o texto do PDF (os arquivos têm texto de verdade, não são imagens).

A leitura é lenta (de 15 a 45 segundos por PDF, **uns 7 minutos no total**), então o texto de cada página é
lido **uma única vez** aqui e guardado em `texto_paginas`, que é reaproveitado na seção 5.

Depois olhamos o título de cada página para saber **qual repasse** está em cada uma.

In [ ]:
!pip install -q pdfplumber   # descomente no Colab, se a biblioteca não estiver instalada
import pdfplumber

texto_paginas = {}                      # {ano: [texto da página 1, texto da página 2, ...]}
for _, linha in pdfs.iterrows():
    inicio = time.time()
    with pdfplumber.open(linha['caminho']) as pdf:
        texto_paginas[int(linha['ano'])] = [p.extract_text() or '' for p in pdf.pages]
    print(f"{linha['ano']}: {len(texto_paginas[int(linha['ano'])])} páginas lidas em {time.time() - inicio:.0f}s")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
2004: 3 páginas lidas em 5s
2005: 3 páginas lidas em 4s
2006: 3 páginas lidas em 4s
2007: 3 páginas lidas em 5s
2008: 3 páginas lidas em 4s
2009: 3 páginas lidas em 4s
2010: 3 páginas lidas em 5s
2011: 3 páginas lidas em 4s
2012: 18 páginas lidas em 21s
2013: 15 páginas lidas em 17s
2014: 15 páginas lidas em 17s
2015: 15 páginas lidas em 18s
2016: 15 páginas lidas em 18s
2017: 15 páginas lidas em 17s
2018: 15 páginas lidas em 22s


In [ ]:
def repasse_da_pagina(texto):
    """Lê as 6 primeiras linhas do texto da página e diz de qual repasse é a tabela que começa nela.
    Usa palavras inteiras (\b) para não confundir, por exemplo, 'IPI' com 'MIPIBU'."""
    topo = ' '.join(texto.splitlines()[:6]).upper()
    if 'REPASSES AOS MUNIC' in topo:
        return 'RESUMO ANUAL'
    if re.search(r'TOTAL DOS REPASSES|ICMS, IPVA', topo):
        return 'TOTAL'
    for tipo in ['ICMS', 'IPVA', 'IPI', 'ROYA']:          # 'ROYA': de 2014 a 2018 o título diz "ROYATIES" (erro de digitação da SEFAZ)
        if re.search(rf'\b{tipo}', topo) and re.search(r'VALORES REPASSADOS|TRANSFERENCIAS', topo):
            return 'ROYALTIES' if tipo == 'ROYA' else tipo
    return None                                   # página de continuação da tabela anterior

inventario = []
for ano, paginas in texto_paginas.items():
    repasses = [repasse_da_pagina(t) for t in paginas]
    inventario.append({'ano': ano, 'paginas': len(repasses),
                       'inicio_de_cada_tabela': ', '.join(f'pág {i + 1}: {r}' for i, r in enumerate(repasses) if r)})
inventario = pd.DataFrame(inventario)
inventario

,ano,paginas,inicio_de_cada_tabela
0,2004,3,pág 1: RESUMO ANUAL
1,2005,3,pág 1: RESUMO ANUAL
2,2006,3,pág 1: RESUMO ANUAL
3,2007,3,pág 1: RESUMO ANUAL
4,2008,3,pág 1: RESUMO ANUAL
5,2009,3,pág 1: RESUMO ANUAL
6,2010,3,pág 1: RESUMO ANUAL
7,2011,3,pág 1: ICMS
8,2012,18,"pág 1: ICMS, pág 4: ICMS, pág 7: IPVA, pág 10: IPI, pág 13: ROYALTIES, pág 16: TOTAL"
9,2013,15,"pág 1: ICMS, pág 4: IPVA, pág 7: IPI, pág 10: ROYALTIES, pág 13: TOTAL"


**O que o inventário mostra: três formatos de PDF**

| Anos | Formato | Repasses disponíveis |
|---|---|---|
| 2004 a 2010 | **Resumo anual:** uma única tabela com, para cada repasse, novembro, dezembro e **total do ano** | ICMS, IPI, IPVA, ROYALTIES (+ CIDE em alguns anos, que descartamos) |
| 2011 | **Só ICMS**, com os 12 meses e o total | **apenas ICMS** |
| 2012 a 2018 | **Uma tabela por repasse**, com os 12 meses (como as planilhas atuais) | ICMS, IPVA, IPI, ROYALTIES |

Detalhes que o inventário também revelou:
- em **2012**, a tabela do ICMS é tão larga que foi cortada em duas: as páginas 1–3 têm os meses e as
  páginas 4–6 têm só a coluna TOTAL;
- de 2014 a 2018 o título dos royalties está escrito "ROYATIES" (erro de digitação no próprio PDF);
- todos os anos de 2012 em diante têm também uma tabela "TOTAL" (soma dos repasses), que não usamos:
  preferimos somar os repasses um a um, como nas planilhas.

Cada formato vai precisar de uma regra própria de leitura na seção 5.

## 5. PDFs → tabela

O `pdfplumber` devolve o texto de cada página, linha por linha. Cada linha de município vira uma tabela em quatro
passos: **5a** extrair o texto, **5b** consertar os números quebrados, **5c** reconhecer o município e
**5d/5e** ler os valores de acordo com o formato do PDF (resumo anual ou tabela por repasse).

### 5a. Extração do texto

Exemplo: a linha de Acari no PDF de 2004 e no de 2014, exatamente como sai do PDF.

In [ ]:
# texto_paginas (lido na seção 4.1) -> lista de (número da página, linha) para cada ano
texto_pdf = {ano: [(i + 1, linha) for i, texto in enumerate(paginas) for linha in texto.splitlines()]
             for ano, paginas in texto_paginas.items()}

for ano in [2004, 2014]:
    print(ano, '→', next(l for _, l in texto_pdf[ano] if l.upper().startswith('ACAR')))

2004 → ACARÍ 0,1732 6 8.303,56 4 9.453,22 587.057,57 308,26 313,98 3 .224,91 712,26 644,81 3 5.316,40 3 .846,11 3.846,11 3 .600,11 4 .357,77 41.641,64 7 2.924,18 54.769,78 671.086,62
2014 → ACARI 0,2472 0,2444% 9 .571,49 163.073,97 207.693,95 247.659,54 174.619,94 209.501,47 250.925,79 164.209,14 218.427,15 247.616,25 184.559,83 243.980,02 271.807,58 2.593.646,09


### 5b. Tratamento dos números quebrados

Na extração, alguns números saem **partidos por um espaço**: `68.303,56` vira `6 8.303,56`, e `1.861.958,93`
vira `1 .861.958,93`. Se lermos cada pedaço como um número, os valores ficam errados e as colunas se deslocam.

**Regra para remontar:** todo valor completo nesses PDFs termina com vírgula e casas decimais (`8.303,56`,
`0,2444`). Então lemos os pedaços da esquerda para a direita e vamos **colando** até formar algo que termine
em `,dd`. Pedaços como `6` ou `1` (sem vírgula) são o começo do próximo número.

Outros casos tratados:
- um `-` sozinho significa **zero** (nenhum repasse naquele mês);
- de 2014 em diante, o índice de participação vem com `%` (`0,2444%`), que é removido;
- números com **3 ou mais casas decimais** são o **índice de participação** do município, não dinheiro, e são
  descartados. Valores em reais têm sempre 2 casas.

In [ ]:
NUMERO_COMPLETO = re.compile(r'^-?[\d.]*\d,\d+$')     # ex.: 8.303,56 | 0,2444 | 1.861.958,93

def juntar_numeros(pedacos):
    """['6', '8.303,56', '-', '1', '.861.958,93'] -> ['68.303,56', '-', '1.861.958,93'].
    Devolve também o que sobrou sem fechar um número (deve ser vazio)."""
    valores, acumulado = [], ''
    for p in pedacos:
        p = p.replace('%', '')
        if not p:
            continue
        if p == '-' and not acumulado:
            valores.append('-')
            continue
        acumulado += p
        if NUMERO_COMPLETO.match(acumulado):
            valores.append(acumulado)
            acumulado = ''
    return valores, acumulado

def para_numero(texto):
    """'1.861.958,93' -> 1861958.93 | '-' -> 0.0"""
    return 0.0 if texto == '-' else float(texto.replace('.', '').replace(',', '.'))

def eh_indice(texto):
    """Índice de participação tem 3+ casas decimais (0,2444); valores em reais têm 2."""
    return texto != '-' and len(texto.split(',')[1]) >= 3

# Demonstração com a linha de Acari de 2004
exemplo = next(l for _, l in texto_pdf[2004] if l.upper().startswith('ACAR'))
pedacos = exemplo.split()[1:]
valores, sobra = juntar_numeros(pedacos)
print('Antes :', pedacos[:8])
print('Depois:', valores[:5])
print('Em número:', [para_numero(v) for v in valores[:5]])

Antes : ['0,1732', '6', '8.303,56', '4', '9.453,22', '587.057,57', '308,26', '313,98']
Depois: ['0,1732', '68.303,56', '49.453,22', '587.057,57', '308,26']
Em número: [0.1732, 68303.56, 49453.22, 587057.57, 308.26]


### 5c. Reconhecimento do município

Cada linha começa com o nome do município e depois vêm os números. O nome vai até o primeiro pedaço que começa
com dígito ou `-`.

Os PDFs de **2004 a 2008** usam **nomes abreviados** (`CEL. J. PESSOA`, `S. G. AMARANTE`, `FERND.PEDROZA`...) e
os de royalties de **2015 a 2018** trazem `BOM JESUS+A29` (sobra de uma fórmula de planilha colada no PDF).
Todos foram conferidos um a um e acrescentados ao `DE_PARA`, sempre apontando para o nome oficial do IBGE.

In [ ]:
DE_PARA.update({
    # abreviações dos PDFs de 2004 a 2008
    'ALTORODRIGUES': 'ALTODORODRIGUES',
    'CAICARARVENTOS': 'CAICARADORIODOVENTO',
    'CELJPESSOA': 'CORONELJOAOPESSOA',
    'DRSEVERIANO': 'DOUTORSEVERIANO',
    'FCODANTAS': 'FRANCISCODANTAS',
    'FERNDPEDROZA': 'FERNANDOPEDROZA',
    'GOVDSROSADO': 'GOVERNADORDIXSEPTROSADO',
    'OLHODADOBORGES': 'OLHODAGUADOBORGES',
    'RIACHOSANTANA': 'RIACHODESANTANA',
    'RODFERNANDES': 'RODOLFOFERNANDES',
    'SBENTONORTE': 'SAOBENTODONORTE',
    'SBENTOTRAIRI': 'SAOBENTODOTRAIRI',
    'SFRANCISCOOESTE': 'SAOFRANCISCODOOESTE',
    'SGAMARANTE': 'SAOGONCALODOAMARANTE',
    'SJOAOSABUGI': 'SAOJOAODOSABUGI',
    'SJOSECAMPESTRE': 'SAOJOSEDOCAMPESTRE',
    'SJOSEDEMIPIBU': 'SAOJOSEDEMIPIBU',
    'SJOSESERIDO': 'SAOJOSEDOSERIDO',
    'SMIGUELDOGOSTOSO': 'SAOMIGUELDOGOSTOSO',
    'SPAULODOPOTENGI': 'SAOPAULODOPOTENGI',
    'SENELOISOUZA': 'SENADORELOIDESOUZA',
    'SENGEORAVELINO': 'SENADORGEORGINOAVELINO',
    'SERRASBENTO': 'SERRADESAOBENTO',
    'TENLAURENTCRUZ': 'TENENTELAURENTINOCRUZ',
    # sobra de fórmula nos royalties de 2015 a 2018 ("BOM JESUS+A29")
    'BOMJESUSA': 'BOMJESUS',
})
assert set(DE_PARA.values()) <= set(COD_POR_CHAVE), 'algum destino do DE_PARA não existe na lista do IBGE'

# Linhas que não são municípios (títulos, totais, observações)
NAO_MUNICIPIO = re.compile(r'^(TOTA|OBS|REPASSE|PUBLICA|ATUALIZ|MUNIC|COORD|VALORES|TRANSF|INDICE|GOVERNO|SECRETARIA)')

def ler_linha(linha):
    """Separa o nome dos números e devolve (nome, código IBGE ou None, valores em reais, sobra)."""
    pedacos = linha.split()
    i = 0
    while i < len(pedacos) and not re.match(r'^[-\d.]', pedacos[i]):
        i += 1
    nome = ' '.join(pedacos[:i])
    valores, sobra = juntar_numeros(pedacos[i:])
    reais = [para_numero(v) for v in valores if not eh_indice(v)]
    k = DE_PARA.get(chave(nome), chave(nome))
    return nome, COD_POR_CHAVE.get(k), reais, sobra

### 5d. Formato "resumo anual" (2004 a 2010)

Uma tabela só, com um bloco de colunas por repasse. Cada bloco termina em **TOTAL DO ANO**, que é o valor que
usamos. O código lê os títulos da tabela para saber a ordem dos blocos (em 2004 há um bloco de **CIDE**, que é
descartado) e quantas colunas cada um tem.

**Conferências:**
- em cada linha, ICMS + IPI + IPVA + (CIDE) + ROYALTIES tem de ser igual ao "TOTAL DOS REPASSES" do próprio PDF;
- a soma dos 167 municípios é comparada com a linha de total do estado (primeira linha de números da tabela).

In [ ]:
MESES_PDF = r'TOTAL DO ANO|JANEIRO|FEVEREIRO|MAR.O|ABRIL|MAIO|JUNHO|JULHO|AGOSTO|SETEMBRO|OUTUBRO|NOVEMBRO|DEZEMBRO'

def ler_resumo_anual(ano):
    linhas = [l for _, l in texto_pdf[ano]]
    linha_impostos = next(l.upper() for l in linhas if re.search(r'\bICMS\b.*\bROYALTIES\b', l.upper()))
    impostos = re.findall(r'ICMS|IPVA|IPI|CIDE|ROYALTIES|TOTAL DOS REPASSES', linha_impostos)
    linha_colunas = next(l.upper() for l in linhas if 'TOTAL DO ANO' in l.upper())
    colunas = re.findall(MESES_PDF, linha_colunas)

    # a que repasse pertence cada coluna: muda de bloco a cada "TOTAL DO ANO"
    bloco, dono = 0, []
    for c in colunas:
        dono.append(impostos[bloco])
        if c == 'TOTAL DO ANO':
            bloco += 1
    posicao_total = {dono[i]: i for i, c in enumerate(colunas) if c == 'TOTAL DO ANO'}

    registros, problemas, linha_estado = [], [], None
    for linha in linhas:
        nome, cod, reais, sobra = ler_linha(linha)
        if not nome and len(reais) == len(colunas) and linha_estado is None:
            linha_estado = {t: reais[p] for t, p in posicao_total.items()}
        if cod is None:
            if nome and reais and not NAO_MUNICIPIO.match(chave(nome)):
                problemas.append(f'nome não reconhecido: {nome}')
            continue
        if len(reais) != len(colunas) or sobra:
            problemas.append(f'{nome}: {len(reais)} valores, esperados {len(colunas)}')
            continue
        totais = {t: reais[p] for t, p in posicao_total.items()}
        soma = sum(v for t, v in totais.items() if t != 'TOTAL DOS REPASSES')
        for tipo in TIPOS:
            registros.append({'cod_ibge': cod, 'nome_na_fonte': nome, 'ano': ano, 'tipo': tipo,
                              'valor': totais[tipo], 'dif_linha': soma - totais['TOTAL DOS REPASSES']})
    return pd.DataFrame(registros), linha_estado, problemas, impostos

partes_resumo, diag_pdf = [], []
for ano in range(2004, 2011):
    tabela, estado, problemas, impostos = ler_resumo_anual(ano)
    partes_resumo.append(tabela)
    for tipo in TIPOS:
        t = tabela[tabela['tipo'] == tipo]
        diag_pdf.append({'ano': ano, 'tipo': tipo, 'formato': 'resumo anual', 'municipios': t['cod_ibge'].nunique(),
                         'maior_dif_linha_R$': t['dif_linha'].abs().max(),
                         'soma_municipios': t['valor'].sum(), 'total_estado_pdf': estado.get(tipo) if estado else np.nan,
                         'problemas': problemas})
    print(ano, '| blocos:', ', '.join(impostos), '| problemas:', problemas or 'nenhum')
repasses_resumo = pd.concat(partes_resumo, ignore_index=True)

2004 | blocos: ICMS, IPI, IPVA, CIDE, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2005 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2006 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2007 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2008 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2009 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum
2010 | blocos: ICMS, IPI, IPVA, ROYALTIES, TOTAL DOS REPASSES | problemas: nenhum


### 5e. Formato "tabela por repasse" (2011 a 2018)

Uma tabela para cada repasse (ICMS, IPVA, IPI, ROYALTIES), cada uma ocupando algumas páginas, com os 12 meses
(mais a parcela de janeiro pelo índice do ano anterior, no ICMS) e, quase sempre, uma coluna **TOTAL**.

- Usamos a seção 4.1 (`repasse_da_pagina`) para saber onde cada tabela começa; as páginas seguintes sem título
  são continuação. A tabela "TOTAL" (soma dos repasses) é ignorada.
- Se a tabela tem coluna TOTAL, o último valor da linha é o total do ano, e conferimos que ele é igual à soma
  dos meses. Se não tem (ICMS de 2012, cuja coluna TOTAL foi para outras páginas, sem nomes), o total do ano é a
  soma dos meses.
- **2011:** o PDF só tem ICMS. IPVA, IPI e ROYALTIES desse ano ficam **sem dado** (decisão do grupo: manter 2011
  para não quebrar a série, e sinalizar nas análises que nesse ano só há ICMS).

In [ ]:
def tabelas_por_repasse(ano):
    """Agrupa as linhas do PDF por tabela: {'ICMS': [linhas], 'IPVA': [linhas], ...}."""
    tipos_pagina = [repasse_da_pagina(t) for t in texto_paginas[ano]]
    tabelas, atual = {}, None
    for (pagina, linha) in texto_pdf[ano]:
        tipo = tipos_pagina[pagina - 1]
        if tipo:
            atual = tipo
        tabelas.setdefault(atual, []).append(linha)
    return tabelas

partes_mensal = []
for ano in range(2011, 2019):
    for tipo, linhas in tabelas_por_repasse(ano).items():
        if tipo not in TIPOS:
            continue
        # cabeçalho = linhas antes do primeiro município
        primeira = next(i for i, l in enumerate(linhas) if ler_linha(l)[1] is not None)
        tem_total = 'TOTAL' in ' '.join(linhas[:primeira]).upper()

        registros, problemas, linha_estado = [], [], None
        for linha in linhas:
            nome, cod, reais, sobra = ler_linha(linha)
            if not nome and reais and linha_estado is None:
                linha_estado = reais[-1] if tem_total else sum(reais)
            if cod is None:
                if nome and reais and not NAO_MUNICIPIO.match(chave(nome)):
                    problemas.append(f'nome não reconhecido: {nome}')
                continue
            if sobra or not reais:
                problemas.append(f'{nome}: número incompleto')
                continue
            total = reais[-1] if tem_total else sum(reais)
            soma_meses = sum(reais[:-1]) if tem_total else total
            registros.append({'cod_ibge': cod, 'nome_na_fonte': nome, 'ano': ano, 'tipo': tipo,
                              'valor': total, 'dif_linha': soma_meses - total})
        tabela = pd.DataFrame(registros)
        partes_mensal.append(tabela)
        diag_pdf.append({'ano': ano, 'tipo': tipo, 'formato': 'tabela por repasse',
                         'municipios': tabela['cod_ibge'].nunique(),
                         'maior_dif_linha_R$': tabela['dif_linha'].abs().max(),
                         'soma_municipios': tabela['valor'].sum(), 'total_estado_pdf': linha_estado,
                         'problemas': problemas})
repasses_mensal = pd.concat(partes_mensal, ignore_index=True)

### 5f. Conferência dos PDFs

In [ ]:
diag_pdf = pd.DataFrame(diag_pdf)
diag_pdf['dif_estado_%'] = (diag_pdf['soma_municipios'] / diag_pdf['total_estado_pdf'] - 1) * 100
(diag_pdf.assign(soma_R_milhoes=diag_pdf['soma_municipios'] / 1e6)
 [['ano', 'tipo', 'formato', 'municipios', 'maior_dif_linha_R$', 'soma_R_milhoes', 'dif_estado_%']]
 .round(3))

,ano,tipo,formato,municipios,maior_dif_linha_R$,soma_R_milhoes,dif_estado_%
0,2004,ICMS,resumo anual,167,0.02,338.797,0.000
1,2004,IPVA,resumo anual,167,0.02,27.651,0.000
2,2004,IPI,resumo anual,167,0.02,1.862,-0.000
3,2004,ROYALTIES,resumo anual,167,0.02,24.043,0.000
4,2005,ICMS,resumo anual,167,0.01,398.069,0.000
5,2005,IPVA,resumo anual,167,0.01,45.840,-0.000
6,2005,IPI,resumo anual,167,0.01,5.478,-0.000
7,2005,ROYALTIES,resumo anual,167,0.01,29.299,0.000
8,2006,ICMS,resumo anual,167,0.02,473.281,0.000
9,2006,IPVA,resumo anual,167,0.02,42.204,0.000


In [ ]:
problemas = [f"{a} {t}: {p}" for a, t, lista in zip(diag_pdf['ano'], diag_pdf['tipo'], diag_pdf['problemas'])
             for p in lista]
assert not problemas, f'linhas com nome não reconhecido ou número incompleto: {problemas}'
assert (diag_pdf['municipios'] == 167).all(), 'alguma tabela não tem os 167 municípios'
assert (diag_pdf['maior_dif_linha_R$'] < 5).all(), 'total de algum município não bate com a soma das partes'
assert (diag_pdf['dif_estado_%'].abs() < 0.5).all(), 'soma dos municípios muito diferente do total do estado'
print('OK: 167 municípios em todas as tabelas, nenhum nome desconhecido, totais conferidos')

OK: 167 municípios em todas as tabelas, nenhum nome desconhecido, totais conferidos


**Leitura da conferência**
- `maior_dif_linha_R$`: maior diferença, entre todos os municípios, entre o total da linha e a soma das partes
  (meses, ou repasses no resumo anual). Fica abaixo de R$ 3: é arredondamento do próprio PDF.
- `dif_estado_%`: soma dos 167 municípios × linha de total do estado no PDF. Fica em zero em todos os anos e
  tipos, exceto **2008** (−0,008%, ou seja, alguns milhares de reais em centenas de milhões: arredondamento do
  próprio PDF).
- **2011** aparece só com ICMS, como esperado.

In [ ]:
repasses_pdf = pd.concat([repasses_resumo, repasses_mensal], ignore_index=True).drop(columns='dif_linha')
repasses_pdf['formato_fonte'] = 'pdf'
assert not repasses_pdf.duplicated(['cod_ibge', 'ano', 'tipo']).any(), 'há município repetido'

# Total anual por tipo extraído dos PDFs (R$ milhões, valores nominais)
(repasses_pdf.pivot_table(index='ano', columns='tipo', values='valor', aggfunc='sum') / 1e6).round(1)

tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2004,338.8,1.9,27.7,24.0
2005,398.1,5.5,45.8,29.3
2006,473.3,3.5,42.2,26.8
2007,527.2,1.9,56.0,24.0
2008,561.2,1.1,60.4,31.8
2009,587.2,1.2,70.3,20.8
2010,678.0,1.2,74.4,23.7
2011,778.2,NaN,NaN,NaN
2012,890.8,0.9,96.7,37.2


## 6. Junção de planilhas + PDFs e conferência

### 6.1 Série completa

Juntamos as planilhas (2019–2025) e os PDFs (2004–2018) numa tabela no mesmo formato do notebook 01:
**uma linha por município, ano e tipo de repasse**.

Montamos a **grade completa** (167 municípios × 22 anos × 4 tipos) para que nenhuma combinação "suma" sem
aviso. O único buraco esperado é 2011 (IPVA, IPI e ROYALTIES), que fica com valor vazio e uma observação.

In [ ]:
repasses = pd.concat([repasses_pdf, repasses_xls], ignore_index=True)

anos_serie = list(range(int(escolhidos['ano'].min()), int(escolhidos['ano'].max()) + 1))
grade = (municipios[['cod_ibge', 'municipio']]
         .merge(pd.DataFrame({'ano': anos_serie}), how='cross')
         .merge(pd.DataFrame({'tipo': TIPOS}), how='cross'))

serie = grade.merge(repasses[['cod_ibge', 'ano', 'tipo', 'valor', 'formato_fonte']],
                    on=['cod_ibge', 'ano', 'tipo'], how='left')
serie['observacao'] = np.where((serie['ano'] == 2011) & (serie['tipo'] != 'ICMS'),
                               'não publicado pela SEFAZ (em 2011 o PDF só tem ICMS)', '')
serie = serie.sort_values(['ano', 'tipo', 'cod_ibge']).reset_index(drop=True)
print(serie.shape, '→ esperado', 167 * len(anos_serie) * len(TIPOS), 'linhas')

(14696, 7) → esperado 14696 linhas


In [ ]:
# Conferências da série
faltando = serie[serie['valor'].isna()]
assert len(serie) == 167 * len(anos_serie) * len(TIPOS), 'grade incompleta'
assert not serie.duplicated(['cod_ibge', 'ano', 'tipo']).any(), 'combinação repetida'
assert set(zip(faltando['ano'], faltando['tipo'])) == {(2011, 'IPVA'), (2011, 'IPI'), (2011, 'ROYALTIES')}, \
    'há valores faltando fora do caso conhecido de 2011'
assert (serie['valor'].dropna() >= 0).all(), 'há valor negativo'
print(f'OK: {len(serie):,} linhas; só faltam os {len(faltando)} valores conhecidos de 2011 (3 tipos × 167 municípios)')

OK: 14,696 linhas; só faltam os 501 valores conhecidos de 2011 (3 tipos × 167 municípios)


### 6.2 Continuidade da série

Total do estado por ano (R$ milhões, valores nominais) e variação em relação ao ano anterior. Serve para ver se a
troca de fonte (PDF até 2018, planilha a partir de 2019) criou algum salto artificial.

In [ ]:
por_ano = serie.groupby(['ano', 'tipo'])['valor'].sum(min_count=1).unstack()[TIPOS] / 1e6   # min_count=1: vazio continua vazio
por_ano['TOTAL (4 tipos)'] = por_ano[TIPOS].sum(axis=1, min_count=4)        # 2011 fica vazio de propósito
por_ano['fonte'] = serie.groupby('ano')['formato_fonte'].first()
por_ano['variação ICMS %'] = por_ano['ICMS'].pct_change() * 100
por_ano.round(1)

tipo,ICMS,IPVA,IPI,ROYALTIES,TOTAL (4 tipos),fonte,variação ICMS %
ano,,,,,,,
2004,338.8,27.7,1.9,24.0,392.4,pdf,NaN
2005,398.1,45.8,5.5,29.3,478.7,pdf,17.5
2006,473.3,42.2,3.5,26.8,545.7,pdf,18.9
2007,527.2,56.0,1.9,24.0,609.1,pdf,11.4
2008,561.2,60.4,1.1,31.8,654.5,pdf,6.4
2009,587.2,70.3,1.2,20.8,679.6,pdf,4.6
2010,678.0,74.4,1.2,23.7,777.3,pdf,15.5
2011,778.2,NaN,NaN,NaN,NaN,pdf,14.8
2012,890.8,96.7,0.9,37.2,1025.5,pdf,14.5


**Leitura:**
- Na passagem de 2018 (PDF) para 2019 (planilha), o ICMS sobe só cerca de 1%: não há salto artificial pela troca
  de fonte.
- **2011** aparece só com ICMS; o total dos 4 tipos fica vazio de propósito. Nas análises e gráficos, 2011 entra
  nas séries de ICMS e deve ser **sinalizado** (ou deixado de fora) quando o assunto for o total dos repasses.
- Os valores ainda são **nominais**. A correção pela inflação (IPCA) será feita junto com a ampliação dos outros
  indicadores (população, PIB, CadÚnico), na próxima etapa.

## 7. Saída

Mesmo formato do `repasses_2020_2023.csv` do notebook 01 (uma linha por município, ano e tipo), agora de
**2004 a 2025**, com três colunas a mais: `municipio`, `formato_fonte` e `observacao`.

In [ ]:
nome_saida = f'saida/repasses_{anos_serie[0]}_{anos_serie[-1]}.csv'
serie.to_csv(nome_saida, index=False)

dicionario_repasses = pd.DataFrame([
    ['cod_ibge', 'Código IBGE do município (chave de integração)', 'IBGE', '-'],
    ['municipio', 'Nome oficial do município', 'IBGE', '-'],
    ['ano', f'Ano do repasse ({anos_serie[0]} a {anos_serie[-1]})', '-', 'ano'],
    ['tipo', 'Tipo de repasse: ICMS (25%), IPVA (50%), IPI-Exportação (25%), ROYALTIES (25%)', 'SEFAZ/RN', 'categoria'],
    ['valor', 'Valor repassado ao município no ano (vazio em 2011 para IPVA, IPI e ROYALTIES)', 'SEFAZ/RN', 'R$ nominais'],
    ['formato_fonte', 'Arquivo de origem: pdf (2004–2018) ou xls (2019–2025)', 'SEFAZ/RN', 'categoria'],
    ['observacao', 'Aviso sobre o dado (preenchido só nos casos sem publicação)', 'Derivada', 'texto'],
], columns=['variável', 'descrição', 'fonte', 'unidade'])
dicionario_repasses['tipo_dado'] = dicionario_repasses['variável'].map(serie.dtypes.astype(str))
dicionario_repasses.to_csv(f'saida/dicionario_repasses_{anos_serie[0]}_{anos_serie[-1]}.csv', index=False)

print('Arquivos salvos:', nome_saida, f'e saida/dicionario_repasses_{anos_serie[0]}_{anos_serie[-1]}.csv')
dicionario_repasses

Arquivos salvos: saida/repasses_2004_2025.csv e saida/dicionario_repasses_2004_2025.csv


,variável,descrição,fonte,unidade,tipo_dado
0,cod_ibge,Código IBGE do município (chave de integração),IBGE,-,int64
1,municipio,Nome oficial do município,IBGE,-,object
2,ano,Ano do repasse (2004 a 2025),-,ano,int64
3,tipo,"Tipo de repasse: ICMS (25%), IPVA (50%), IPI-Exportação (25%), ROYALTIES (25%)",SEFAZ/RN,categoria,object
4,valor,"Valor repassado ao município no ano (vazio em 2011 para IPVA, IPI e ROYALTIES)",SEFAZ/RN,R$ nominais,float64
5,formato_fonte,Arquivo de origem: pdf (2004–2018) ou xls (2019–2025),SEFAZ/RN,categoria,object
6,observacao,Aviso sobre o dado (preenchido só nos casos sem publicação),Derivada,texto,object


## Resumo do notebook

- **Coleta automática:** o notebook mapeia o site da SEFAZ, escolhe um arquivo por ano (o mais recente) e baixa
  cada um, primeiro da cópia no GitHub e, se faltar, do site.
- **Série de 22 anos (2004–2025):** planilhas de 2019 a 2025 e PDFs de 2004 a 2018, todos com os 167 municípios.
- **Problemas encontrados e tratados:** três formatos diferentes de PDF, números partidos na extração, 24 nomes
  abreviados e uma sobra de fórmula (`BOM JESUS+A29`), erro de digitação no título ("ROYATIES"), tabela de ICMS
  de 2012 cortada em duas, e a parcela "INDICE ano anterior JANEIRO" do ICMS, que também corrigiu o notebook 01.
- **Validações:** soma dos meses = coluna TOTAL em todas as planilhas; total de cada município = soma das partes
  nos PDFs; soma dos municípios = total do estado.
- **Limitação:** em 2011 a SEFAZ só publicou o ICMS.